In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
column_names = [
    "duration", "protocol_type", "service", "flag", "src_bytes",
    "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty"
]

In [ ]:
train_path = "/kaggle/input/datasets/hassan06/nslkdd/KDDTrain+.txt"
test_path = "/kaggle/input/datasets/hassan06/nslkdd/KDDTest+.txt"

train_df = pd.read_csv(train_path, names=column_names)
test_df = pd.read_csv(test_path, names=column_names)

print("شکل (Shape) دیتای ترین:", train_df.shape)
print("شکل (Shape) دیتای تست:", test_df.shape)

In [ ]:
train_path = "/kaggle/input/nslkdd/KDDTrain+.txt"
test_path = "/kaggle/input/nslkdd/KDDTest+.txt"

In [ ]:
train_df.head()

In [ ]:
# دیکشنری (Dictionary) که هر نام حمله رو به دسته‌ی اصلیش نگاشت (Map) می‌کنه
attack_mapping = {
    'normal': 'normal',

    # دسته‌ی DoS
    'neptune': 'DoS', 'back': 'DoS', 'land': 'DoS', 'pod': 'DoS',
    'smurf': 'DoS', 'teardrop': 'DoS', 'mailbomb': 'DoS',
    'apache2': 'DoS', 'processtable': 'DoS', 'udpstorm': 'DoS',

    # دسته‌ی Probe
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe',
    'satan': 'Probe', 'mscan': 'Probe', 'saint': 'Probe',

    # دسته‌ی R2L
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L',
    'multihop': 'R2L', 'phf': 'R2L', 'spy': 'R2L',
    'warezclient': 'R2L', 'warezmaster': 'R2L', 'xlock': 'R2L',
    'xsnoop': 'R2L', 'snmpguess': 'R2L', 'snmpgetattack': 'R2L',
    'httptunnel': 'R2L', 'sendmail': 'R2L', 'named': 'R2L',
    'worm': 'R2L',

    # دسته‌ی U2R
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R',
    'rootkit': 'U2R', 'ps': 'U2R', 'sqlattack': 'U2R',
    'xterm': 'U2R'
}

# ساخت یک ستون جدید به اسم attack_category بر اساس این نگاشت
train_df['attack_category'] = train_df['label'].map(attack_mapping)
test_df['attack_category'] = test_df['label'].map(attack_mapping)

# چک کردن نتیجه
print(train_df['attack_category'].value_counts())

In [ ]:
print("مقادیر خالی در train:", train_df['attack_category'].isnull().sum())
print("مقادیر خالی در test:", test_df['attack_category'].isnull().sum())

In [ ]:
print(train_df['protocol_type'].value_counts())

In [ ]:
print(train_df['service'].value_counts().head(15))

In [ ]:
print(train_df['flag'].value_counts())

In [ ]:
pd.crosstab(train_df['flag'], train_df['attack_category'])

In [ ]:
pd.crosstab(train_df['protocol_type'], train_df['attack_category']).plot(kind='bar', stacked=True, figsize=(10,6))
plt.title('Correlation between Protocol Type and attack Category')
plt.xlabel('protocoltype ')
plt.ylabel('number')
plt.show()

In [ ]:
train_df[['duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count']].describe()

In [ ]:
train_df.groupby('attack_category')[['duration', 'src_bytes', 'dst_bytes', 'count']].mean()

In [ ]:
# بخش یک: تبدیل ستون‌های دسته‌ای به عدد
categorical_columns = ['protocol_type', 'service', 'flag']
train_encoded = pd.get_dummies(train_df, columns=categorical_columns)
test_encoded = pd.get_dummies(test_df, columns=categorical_columns)

# بخش دو: هماهنگ کردن ستون‌های ترین و تست
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

# بخش سه: جدا کردن ویژگی‌ها از برچسب
X_train = train_encoded.drop(['label', 'attack_category', 'difficulty'], axis=1)
y_train = train_encoded['attack_category']

X_test = test_encoded.drop(['label', 'attack_category', 'difficulty'], axis=1)
y_test = test_encoded['attack_category']

# بخش چهار: مقیاس‌بندی ویژگی‌های عددی
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# بررسی نهایی
print("شکل X_train:", X_train.shape)
print("شکل X_test:", X_test.shape)
print("شکل y_train:", y_train.shape)
print("شکل y_test:", y_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,          # تعداد درخت‌های داخل جنگل
    class_weight='balanced',   # مدیریت داده‌ی نامتوازن
    random_state=42,          # برای اینکه نتیجه هر بار یکسان باشه
    n_jobs=-1                # استفاده از همه‌ی هسته‌های پردازنده برای سرعت بیشتر
)

rf_model.fit(X_train_scaled, y_train)
print("اموزش مدل تمام شد")

y_pred = rf_model.predict(X_test_scaled)


from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

print("===گزارش کامل ارزیابی===")
print (classification_report(y_test,y_pred))

print("\n===ماتریس درهم ریختگی===")
cm = confusion_matrix(y_test,y_pred,labels=rf_model.classes_)
cm_df=pd.DataFrame(cm, index=rf_model.classes_,columns=rf_model.classes_)
print (cm_df)

feature_importance = pd.DataFrame({
    'feature':X_train.columns,
'importance': rf_model.feature_importances_
}).sort_values('importance',ascending=False)

print("\n======")
print(feature_importance.head(10))


In [ ]:
!pip install imbalanced-learn --quiet

sampling_strategy_dict={
    'U2R':2000,
    'R2L':5000,
    'Probe':15000
}

from imblearn.over_sampling import SMOTE

smote_custom = SMOTE(
    sampling_strategy=sampling_strategy_dict,
    random_state=42,
    k_neighbors=5
)

print ("before SMOTE:")
print (y_train.value_counts())


X_train_smote, y_train_smote=smote_custom.fit_resample(X_train_scaled,y_train)

print ("\n after SMOTE:")
print (y_train_smote.value_counts())

from sklearn.ensemble import RandomForestClassifier
rf_model_v2 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    max_depth=20,
    min_samples_leaf=2,
    
)

rf_model_v2.fit(X_train_smote,y_train_smote)
print ("\n end train new model")

y_pred_v2= rf_model_v2.predict(X_test_scaled)

from sklearn.metrics import classification_report
print("\n=== گزارش ارزیابی مدل جدید  (After SMOTE) ===")
print(classification_report(y_test, y_pred_v2))



In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

le= LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'

             
)
xgb_model.fit(X_train_scaled,y_train_encoded)
print("آموزش مدل XGBoost تمام شد")

y_pred_xgb= xgb_model.predict(X_test_scaled)

from sklearn.metrics import classification_report


print("=== گزارش ارزیابی مدل XGBoost ===")
print(classification_report(y_test_encoded,y_pred_xgb,target_names=le.classes_))




In [ ]:

y_train_smote_encoded = le.transform(y_train_smote)


from xgboost import XGBClassifier

xgb_model_v2 = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)


xgb_model_v2.fit(X_train_smote, y_train_smote_encoded)
print("آموزش مدل XGBoost + SMOTE تمام شد")


y_pred_xgb_v2 = xgb_model_v2.predict(X_test_scaled)


from sklearn.metrics import classification_report

print("=== گزارش ارزیابی مدل XGBoost + SMOTE ===")
print(classification_report(y_test_encoded, y_pred_xgb_v2, target_names=le.classes_))